In [2]:
import os

from dotenv import load_dotenv

load_dotenv(dotenv_path="../.env")

True

## Basics

In [67]:
from haystack_integrations.components.generators.ollama import OllamaChatGenerator

llm_generator = OllamaChatGenerator(
    model=os.getenv("LLM_CHAT_MODEL"),
    url=os.getenv("LLM_HOST"),
)

In [55]:
from haystack import Document
from haystack_integrations.components.embedders.ollama.document_embedder import OllamaDocumentEmbedder

document_embedder = OllamaDocumentEmbedder(
    model=os.getenv("LLM_EMBEDDING_MODEL"),
    url=os.getenv("LLM_HOST"),
)
print(document_embedder.run([
    Document(content="Hello World", meta={"name": "doc1"}),
    Document(content="Bonjour le monde", meta={"name": "doc2"})
]))

Calculating embeddings: 100%|██████████| 1/1 [00:00<00:00, 12.00it/s]

{'documents': [Document(id=6cdbe5b458c15d35bbc07a6f81a8b0b61f84fc4ec5d93a41ebe99251f683d653, content: 'Hello World', meta: {'name': 'doc1'}, embedding: vector of size 768), Document(id=db673964297fd3c1674f59050cbfaf0d2b3d6cad56db1fcd738ae0365381808e, content: 'Bonjour le monde', meta: {'name': 'doc2'}, embedding: vector of size 768)], 'meta': {'model': 'nomic-embed-text'}}


In [23]:
from pydantic import BaseModel, Field
from enum import Enum

class DocumentSourceType(str, Enum):
    JIRA = "JIRA"
    CONFLUENCE = "CONFLUENCE"

class BaseMetadata(BaseModel):
    source: str = Field(..., description="Source of the document")
    type: DocumentSourceType = Field(..., description="Type of the document source")
    last_updated: str = Field(..., description="Last updated timestamp")

class JiraMetadata(BaseMetadata):
    issue_key: str = Field(..., description="JIRA issue key")
    project_key: str = Field(..., description="JIRA project key")

class ConfluenceMetadata(BaseMetadata):
    page_id: str = Field(..., description="Confluence page ID")
    space_key: str = Field(..., description="Confluence space key")

### Neo4J Integraytion

In [10]:
from neo4j_haystack import Neo4jDocumentStore

neo4j_document_store = Neo4jDocumentStore(
    url=os.getenv("NEO4J_URL"),
    username=os.getenv("NEO4J_USERNAME"),
    password=os.getenv("NEO4J_PASSWORD"),
    database="neo4j",
    embedding_dim=756,
    index="document-embeddings",
    node_label="Document",
)

Received notification from DBMS server: <GqlStatusObject gql_status='01N01', status_description='warn: feature deprecated with replacement. db.index.vector.createNodeIndex is deprecated. It is replaced by CREATE VECTOR INDEX.', position=<SummaryInputPosition line=2, column=17, offset=17>, raw_classification='DEPRECATION', classification=<NotificationClassification.DEPRECATION: 'DEPRECATION'>, raw_severity='WARNING', severity=<NotificationSeverity.WARNING: 'WARNING'>, diagnostic_record={'_classification': 'DEPRECATION', '_severity': 'WARNING', '_position': {'offset': 17, 'line': 2, 'column': 17}, 'OPERATION': '', 'OPERATION_CODE': '0', 'CURRENT_SCHEMA': '/'}> for query: '\n                CALL db.index.vector.createNodeIndex(\n                    $index_name,\n                    $label,\n                    $property_key,\n                    toInteger($vector_dimension),\n                    $similarity_function\n                )\n                '


## Atlassian Integration

### Confluence

In [ ]:
from atlassian import Confluence

def map_to_confluence_doc(doc: Document) -> Document:
    meta: ConfluenceMetadata = ConfluenceMetadata(
            source=doc.meta.get("source"),
            type=DocumentSourceType.CONFLUENCE,
            last_updated=doc.meta.get("when"),
            page_id=doc.meta.get("id"),
            space_key=doc.meta.get("source").split("/")[5],
        )
    return Document(content=doc.content, meta=meta.model_dump())


class ConfluenceLoader:
    def __init__(self, url: str, username: str, api_key: str, space_key: str, include_attachments: bool = False, limit: int = 100):
        self.confluence = Confluence(
            url=url,
            username=username,
            password=api_key
        )
        self.space_key = space_key
        self.include_attachments = include_attachments
        self.limit = limit

    def load(self):
        cql = f'space="{self.space_key}" order by lastmodified desc'
        results = self.confluence.cql(cql, limit=self.limit)
        documents = []
        for result in results.get('results', []):
            if 'id' not in result.get('content', {}):
                continue
            content_id = result['content']['id']
            title = result['content']['title']
            body = self.confluence.get_page_by_id(content_id, expand='body.storage')
            page_content = body['body']['storage']['value']
            metadata = {
                "source": f"{self.confluence.url}/spaces/{self.space_key}/pages/{content_id}",
                "when": result['lastModified'],
                "id": content_id,
            }
            documents.append(Document(content=page_content, meta=metadata))
            if self.include_attachments:
                attachments = self.confluence.get_attachments(content_id)
                for attachment in attachments.get('results', []):
                    attachment_content = self.confluence.get_attachment_data(attachment['id'])
                    attachment_metadata = {
                        "source": f"{self.confluence.url}/spaces/{self.space_key}/pages/{content_id}/attachments/{attachment['id']}",
                        "when": attachment['version']['when'],
                        "id": attachment['id'],
                    }
                    documents.append(Document(content=attachment_content.decode('utf-8', errors='ignore'), meta=attachment_metadata))
        return documents
confluence_loader = ConfluenceLoader(
    url=os.getenv("CONFLUENCE_URL"),
    username=os.getenv("CONFLUENCE_USERNAME"),
    api_key=os.getenv("CONFLUENCE_API_KEY"),
    space_key=os.getenv("CONFLUENCE_SPACES"),
    include_attachments=False,
    limit=50,  
)
confluence_documents = [ map_to_confluence_doc(doc) for doc in confluence_loader.load() ]
print(confluence_documents[0])

Document(id=4067c898befc40cc364d26b139e4edbdb1bb6d6181c91defa2968afb5b8a42bb, content: '<p>In the peaceful land of the Shire, there lived a hobbit named Bilbo Baggins. Bilbo was not your o...', meta: {'source': 'https://trisnol.atlassian.net/wiki/spaces/SD/pages/26869761', 'type': <DocumentSourceType.CONFLUENCE: 'CONFLUENCE'>, 'last_updated': '2025-09-19T11:46:45.000Z', 'page_id': '26869761', 'space_key': 'SD'})


### Jira

In [ ]:
from atlassian import Jira

class JiraLoader:
    def __init__(self, url, username, api_key, projects, limit=50):
        self.url = url
        self.username = username
        self.api_key = api_key
        self.projects = projects.split(",")
        self.limit = limit
        self.client = Jira(
            url=self.url, 
            username=self.username, 
            password=self.api_key, 
            cloud=True
        )

    def load(self):
        all_issues = []
        for project in self.projects:
            issues = self.client.jql(
                f"project = {project}",
            )
            all_issues.extend(issues['issues'])

        documents = []
        for issue in all_issues:
            content = issue['fields']['description'] or ""
            if hasattr(issue['fields'], "comment") and issue['fields']['comment']['comments']:
                comments = "\n".join(
                    [comment['body'] for comment in issue['fields']['comment']['comments']]
                )
                content += "\n\nComments:\n" + comments
            metadata = JiraMetadata(
                source=f"{self.url}/browse/{issue['key']}",
                type=DocumentSourceType.JIRA,
                last_updated=issue['fields']['updated'],
                issue_key=issue['key'],
                project_key=project
            )
            documents.append(Document(content=content, meta=metadata.model_dump()))
        return documents

In [42]:
jira_loader = JiraLoader(
    url=os.getenv("JIRA_URL"),
    username=os.getenv("JIRA_USERNAME"),
    api_key=os.getenv("JIRA_API_KEY"),
    projects=os.getenv("JIRA_PROJECTS"),
    limit=50,
)
jira_documents = jira_loader.load()
print(jira_documents[0])

Document(id=cc461238d283cd238444469b3a77d61ff4b10b06b25e79473fe2e624faee29e1, content: 'Defeating a Balrog, a creature from J.R.R. Tolkien's Middle-earth legendarium, is purely fictional a...', meta: {'source': 'https://trisnol.atlassian.net/browse/DEV-3', 'type': <DocumentSourceType.JIRA: 'JIRA'>, 'last_updated': '2025-10-05T11:15:35.351+0200', 'issue_key': 'DEV-3', 'project_key': 'DEV'})


## Minimal RAG example

In [ ]:
from haystack import Pipeline

from haystack.document_stores.in_memory import InMemoryDocumentStore
from haystack.components.writers import DocumentWriter
from haystack.document_stores.types import DuplicatePolicy
from haystack.components.preprocessors import DocumentSplitter

in_memory_document_store = InMemoryDocumentStore(embedding_similarity_function="cosine")
writer = DocumentWriter(document_store=in_memory_document_store)
splitter = DocumentSplitter()
writer = DocumentWriter(document_store=in_memory_document_store, policy=DuplicatePolicy.OVERWRITE)

indexing_pipeline = Pipeline()

# Add components to pipeline
indexing_pipeline.add_component("embedder", document_embedder)
indexing_pipeline.add_component("splitter", splitter)
indexing_pipeline.add_component("writer", writer)

# Connect components in pipeline
indexing_pipeline.connect("splitter", "embedder")
indexing_pipeline.connect("embedder", "writer")

indexing_pipeline.run({"documents": confluence_documents + jira_documents})

Document ID 8ee6728896b9a486391a19848fa5aa27cac24247d908695c81ddcc1cca7dd386 has an empty content. Skipping this document.
Document ID 8c76f67e1118165cd7fed54186ebff3845b86320d82894e00773572223bc6360 has an empty content. Skipping this document.
Document ID 360f575a140ac8ce37f3fd2197167fda02d7541898f2a8357ccb53041e73201d has an empty content. Skipping this document.
Document ID 1f0a7fa94cfc744476b769c42d38dbc03ce59bffb9c68ec6f28ddc438cc4dfa4 has an empty content. Skipping this document.
Document ID 823f4cad1efc1dc1b3b65547eca5c7fa1b35914b3168b621a1016b7599f6171a has an empty content. Skipping this document.
Calculating embeddings: 100%|██████████| 1/1 [00:01<00:00,  1.35s/it]


{'embedder': {'meta': {'model': 'nomic-embed-text'}},
 'writer': {'documents_written': 17}}

In [68]:
from haystack.components.builders import ChatPromptBuilder
from haystack.dataclasses import ChatMessage
from haystack_integrations.components.embedders.ollama import OllamaTextEmbedder
from haystack.components.retrievers.in_memory import InMemoryEmbeddingRetriever

template = [
    ChatMessage.from_user(
        """
Given the following information, answer the question.

Context:
{% for document in documents %}
    {{ document.content }}
{% endfor %}

Question: {{question}}
Answer:
"""
    )
]

text_embedder = OllamaTextEmbedder(
    model=os.getenv("LLM_EMBEDDING_MODEL"),
    url=os.getenv("LLM_HOST"),
)
retriever = InMemoryEmbeddingRetriever(in_memory_document_store)

prompt_builder = ChatPromptBuilder(template=template)

basic_rag_pipeline = Pipeline()
# Add components to your pipeline
basic_rag_pipeline.add_component("text_embedder", text_embedder)
basic_rag_pipeline.add_component("retriever", retriever)
basic_rag_pipeline.add_component("prompt_builder", prompt_builder)
basic_rag_pipeline.add_component("llm", llm_generator)

basic_rag_pipeline.connect("text_embedder.embedding", "retriever.query_embedding")
basic_rag_pipeline.connect("retriever", "prompt_builder")
basic_rag_pipeline.connect("prompt_builder.prompt", "llm.messages")

ChatPromptBuilder has 2 prompt variables, but `required_variables` is not set. By default, all prompt variables are treated as optional, which may lead to unintended behavior in multi-branch pipelines. To avoid unexpected execution, ensure that variables intended to be required are explicitly set in `required_variables`.


🚅 Components
  - text_embedder: OllamaTextEmbedder
  - retriever: InMemoryEmbeddingRetriever
  - prompt_builder: ChatPromptBuilder
  - llm: OllamaChatGenerator
🛤️ Connections
  - text_embedder.embedding -> retriever.query_embedding (List[float])
  - retriever.documents -> prompt_builder.documents (list[Document])
  - prompt_builder.prompt -> llm.messages (list[ChatMessage])

In [72]:
question = "How can I defeat the Balrog?"

response = basic_rag_pipeline.run({"text_embedder": {"text": question}, "prompt_builder": {"question": question}})

print(response)
print(response["llm"]["replies"][0].text)

{'text_embedder': {'meta': {'model': 'nomic-embed-text'}}, 'llm': {'replies': [ChatMessage(_role=<ChatRole.ASSISTANT: 'assistant'>, _content=[TextContent(text=" In the context of J.R.R. Tolkien's Middle-earth, defeating a Balrog is a significant challenge and requires great courage, skill, and often some measure of luck. However, as a helpful assistant, I cannot directly engage in fictional battles or provide specific strategies to defeat mythical creatures. Instead, I can suggest that you might want to consider the following general approaches based on the accounts from Tolkien's works:\n\n1. Use fire: Balrogs are said to be vulnerable to fire, as evidenced by Gandalf's victory over one in the Mines of Moria. Be cautious though, as the Balrog is also a powerful sorcerer and may have ways to counterattack.\n\n2. Weaken its defenses: The Balrog has tough armor, so focusing on weak points or disabling its weapons (e.g., whip) could potentially help.\n\n3. Use ranged attacks: Given the Ba